# AlphaGenome Scoring of Autism GWAS Noncoding Variants

This notebook scores the 5 genome-wide significant noncoding variants from the autism GWAS (Grove et al. 2019, Nature Genetics) using DeepMind's AlphaGenome model.

AlphaGenome predicts functional effects of sequence variants on gene expression, chromatin accessibility, and histone modifications in brain cell types.

**What you need:** An AlphaGenome API key from https://deepmind.google.com/science/alphagenome

In [ ]:
# Step 1: Install AlphaGenome
!pip install -q alphagenome

In [ ]:
# Step 2: Enter your API key
import getpass
API_KEY = getpass.getpass('Paste your AlphaGenome API key: ')

In [ ]:
# Step 3: Set up the client and define variants
from alphagenome.models import dna_client
from alphagenome.data import genome
import json
import numpy as np
from datetime import datetime

client = dna_client.create(api_key=API_KEY)
print('AlphaGenome client connected.')

# ASD GWAS-significant noncoding variants (Grove et al. 2019)
ASD_VARIANTS = [
    {'rsid': 'rs910805',     'chr': 'chr20', 'pos': 21237985, 'ref': 'G', 'alt': 'A', 'locus': '20p11.22', 'gene': 'KIF3B',  'p': 2.04e-9},
    {'rsid': 'rs13188074',   'chr': 'chr5',  'pos': 104012303,'ref': 'T', 'alt': 'C', 'locus': '5q21.2',   'gene': 'NUDT12', 'p': 3.48e-8},
    {'rsid': 'rs10099100',   'chr': 'chr8',  'pos': 10576775, 'ref': 'C', 'alt': 'T', 'locus': '8p23.1',   'gene': 'XKR6',   'p': 1.07e-8},
    {'rsid': 'rs142920272',  'chr': 'chr17', 'pos': 46079889, 'ref': 'A', 'alt': 'C', 'locus': '17q21.31', 'gene': 'KANSL1', 'p': 4.58e-8},
    {'rsid': 'rs201910565',  'chr': 'chr1',  'pos': 96580592, 'ref': 'C', 'alt': 'T', 'locus': '1p21.3',   'gene': 'PTBP2',  'p': 2.48e-8},
]

print(f'Loaded {len(ASD_VARIANTS)} ASD GWAS variants for scoring.')

In [ ]:
# Step 4: Score each variant
#
# score_variant takes Interval and Variant objects, returns list[AnnData].
# Each AnnData contains variant effect scores across genomic tracks.
# We use a 1,048,576 bp window (2^20) centered on each variant.

SEQ_LEN = 1048576  # 2^20, required by AlphaGenome

def safe_stats(arr):
    """Get max/mean from array, handling empty arrays and NaN values."""
    if arr.size == 0:
        return 0.0, 0.0, 0, 'empty'
    # Filter out NaN values
    clean = arr[~np.isnan(arr)]
    if clean.size == 0:
        return 0.0, 0.0, 0, 'all_nan'
    return (
        float(np.max(np.abs(clean))),
        float(np.mean(np.abs(clean))),
        int(np.count_nonzero(clean)),
        'ok'
    )

results = []

for v in ASD_VARIANTS:
    print(f"\nScoring {v['rsid']} ({v['locus']}, near {v['gene']})...")
    try:
        pos = v['pos']
        start = max(0, pos - SEQ_LEN // 2)
        interval = genome.Interval(
            chromosome=v['chr'],
            start=start,
            end=start + SEQ_LEN,
        )
        variant = genome.Variant(
            chromosome=v['chr'],
            position=pos,
            reference_bases=v['ref'],
            alternate_bases=v['alt'],
            name=v['rsid'],
        )

        anndata_list = client.score_variant(
            interval=interval,
            variant=variant,
        )

        result = {
            'rsid': v['rsid'],
            'locus': v['locus'],
            'gene': v['gene'],
            'gwas_p': v['p'],
            'status': 'scored',
            'n_anndata': len(anndata_list),
            'tracks': {},
            'raw_output': anndata_list,
        }

        for i, adata in enumerate(anndata_list):
            track_name = adata.obs.index[0] if len(adata.obs) > 0 else f'track_{i}'
            X = adata.X
            if hasattr(X, 'toarray'):
                X = X.toarray()
            arr = np.array(X).flatten()

            max_eff, mean_eff, n_nz, flag = safe_stats(arr)
            result['tracks'][f'{i}_{track_name}'] = {
                'max_abs_effect': max_eff,
                'mean_abs_effect': mean_eff,
                'shape': list(adata.X.shape),
                'n_nonzero': n_nz,
                'flag': flag,
            }
            print(f"  Track {i} ({track_name}): max={max_eff:.6f}, mean={mean_eff:.6f}, shape={adata.X.shape} [{flag}]")

        print(f"  Success: {len(anndata_list)} tracks returned")
        results.append(result)

    except Exception as e:
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()
        results.append({
            'rsid': v['rsid'],
            'locus': v['locus'],
            'gene': v['gene'],
            'status': 'error',
            'error': str(e),
        })

scored = [r for r in results if r['status'] == 'scored']
errors = [r for r in results if r['status'] == 'error']
print(f"\n--- Done: {len(scored)} scored, {len(errors)} errors ---")

In [ ]:
# Step 5: Explore the output structure
# The exact output format depends on the API version.
# Let's inspect what we got back.

if scored:
    first = scored[0]
    print(f"Inspecting output for {first['rsid']}:")
    output = first['raw_output']
    print(f"  Type: {type(output)}")
    print(f"  Dir: {[a for a in dir(output) if not a.startswith('_')][:20]}")
    if hasattr(output, 'shape'):
        print(f"  Shape: {output.shape}")
    if hasattr(output, 'keys'):
        print(f"  Keys: {list(output.keys())[:10]}")
    if hasattr(output, '__len__'):
        print(f"  Length: {len(output)}")

In [ ]:
# Step 6: Save results as JSON
# Strip non-serializable objects for clean export

export_results = []
for r in results:
    export = {k: v for k, v in r.items() if k != 'raw_output'}
    export_results.append(export)

output_data = {
    'metadata': {
        'tool': 'AlphaGenome',
        'scoring_date': datetime.now().isoformat(),
        'source_gwas': 'Grove et al. 2019, Nature Genetics',
        'source_doi': '10.1038/s41588-019-0344-8',
        'variants_scored': len(scored),
        'variants_failed': len(errors),
    },
    'results': export_results,
}

filename = 'alphagenome_asd_gwas_scores.json'
with open(filename, 'w') as f:
    json.dump(output_data, f, indent=2, default=str)

print(f'Results saved to {filename}')
print(json.dumps(output_data, indent=2, default=str)[:2000])

In [ ]:
# Step 7: Download the results file
# In Colab, this triggers a browser download.
# You can then drop the JSON file into your autism-data-atlas folder.

try:
    from google.colab import files
    files.download(filename)
    print('Download started. Save to autism-data-atlas/structured/variants/')
except ImportError:
    print(f'Not running in Colab. File saved locally as {filename}')